# Day 06 live demos: threads and concurrency

<img src="illustrations/00_cover.png" alt="One courier takes 2.4 seconds. Three couriers finish with the slowest file, in 1.0 second." width="100%">

Twenty-three demos, in slide order, each with a picture of the idea before the code. Run a cell with **Shift+Enter**. This kernel is Python 3.14 with the GIL on.

The number on the navy rail matches the demo. Demos 9, 10, and 23 are the ones to repeat in a terminal with the free-threaded build, `python3.14t`. Demo 7 runs the heartbeat as its own program, so the kernel stays up. Demo 21 writes `work.py` first, because a process pool can only call a function it can import.


## 1. Three downloads, one at a time

<img src="illustrations/01_sequential.png" alt="One courier carries a.csv, then b.csv, then c.csv. The waits add to 2.4 seconds." width="100%">

*Slide: The cost of waiting* · script `01_sequential.py`

One courier carries `a.csv`, then `b.csv`, then `c.csv`. The sleeps are 1.0, 0.6, and 0.8 seconds, and the clock adds them: **2.4 seconds**. The sleep stands in for a network wait, and the processor is idle the whole time. Hold onto 2.4. The next cell overlaps the same three waits.


In [1]:
import time

def download(name, seconds):
    print("start", name)
    time.sleep(seconds)          # waiting for the network
    print("done ", name)

# seconds each download takes
files = {"a.csv": 1.0, "b.csv": 0.6, "c.csv": 0.8}
start = time.perf_counter()
for name, seconds in files.items():
    download(name, seconds)
print(f"total: {time.perf_counter() - start:.1f} s")      # 2.4 s

start a.csv
done  a.csv
start b.csv
done  b.csv
start c.csv
done  c.csv
total: 2.4 s


## 2. The same downloads in threads

<img src="illustrations/02_threads.png" alt="Three downloads start together. The clock keeps the slowest wait, 1.0 second." width="100%">

*Slide: Three downloads, three threads, one second* · script `02_threads.py`

Three couriers leave together. `start()` returns at once, and `join()` holds the main thread until the last file lands. The clock keeps the slowest wait, **1.0 second**. The done-lines usually arrive b, then c, then a, following the sleep times.


In [2]:
import threading, time

def download(name, seconds):
    time.sleep(seconds)
    print("done ", name)

files = {"a.csv": 1.0, "b.csv": 0.6, "c.csv": 0.8}
start = time.perf_counter()
threads = []
for name, secs in files.items():
    t = threading.Thread(target=download, args=(name, secs))
    t.start()                    # runs download() in a new thread
    print("start", name)
    threads.append(t)
for t in threads:
    t.join()                     # wait until t has finished
print(f"total: {time.perf_counter() - start:.1f} s")      # 1.0 s, b then c then a

start a.csv
start b.csv
start c.csv
done  b.csv
done  c.csv
done  a.csv
total: 1.0 s


## 3. With and without join()

<img src="illustrations/03_join.png" alt="Without join, main speaks while the worker sleeps. With join, main speaks after the worker finishes." width="100%">

*Slide: join(): wait for a thread to finish* · script `03_join.py`

`start()` sends the worker off. `join()` is the main thread waiting at the gate. This cell has `USE_JOIN = True`, so `worker: finished` prints and then main asks `all done?`.

Set `USE_JOIN` to `False` and run the cell again. Main prints the question during the half-second sleep. The kernel keeps running between cells, so a late `worker: finished` can show up under the next cell.


In [3]:
import threading, time

USE_JOIN = True      # change to True and run again

def work():
    time.sleep(0.5)
    print("worker: finished")

t = threading.Thread(target=work)
t.start()
if USE_JOIN:
    t.join()
print("main: all done?")

worker: finished
main: all done?


## 4. Names, current_thread() and active_count()

<img src="illustrations/04_names.png" alt="MainThread plus three named workers, each with a badge." width="100%">

*Slide: Who am I? Naming and inspecting threads* · script `04_thread_info.py`

Every program starts as `MainThread`. The three workers run the same function under different names, which is what makes a log readable. In a notebook, `active_count()` also counts Jupyter's own threads, so the number is higher than the 4 a plain script would print.

The joins at the bottom of the cell are commented out, so the workers can still be alive when the count is printed.


In [4]:
import threading, time

def task():
    me = threading.current_thread()
    print(f"{me.name} running")
    time.sleep(0.2)

print(threading.current_thread().name)
workers = [threading.Thread(target=task, name=f"worker-{i}") for i in range(3)]
print(workers)
for w in workers:
    w.start()
    time.sleep(0.01)
print("alive:", threading.active_count())                 # 4: main + 3 workers
# for w in workers:
#     w.join()
# print("alive:", threading.active_count(), workers[0].is_alive())

MainThread
[<Thread(worker-0, initial)>, <Thread(worker-1, initial)>, <Thread(worker-2, initial)>]
worker-0 running
worker-1 running
worker-2 running
alive: 11


## 5. A thread's return value is lost

<img src="illustrations/05_results.png" alt="join returns None. Three private slots keep the squares 9, 25, and 49." width="100%">

*Slide: Getting results back from a thread* · script `05_results.py`

`join()` returns `None`. `square(4)` does compute 16, and that return value ends with the worker. Give each thread its own slot in a list. The workers write indices 0, 1, and 2, and the list comes back `[9, 25, 49]`.


In [27]:
import threading, time

def square(n):
    return n * n

t = threading.Thread(target=square, args=[4])
t.start()
print("join returned:", t.join())                         # None: the 16 is lost

def square_into(n, results, i):
    time.sleep(0.1)
    results[i] = n * n           # each thread writes its own slot
#
numbers = [3, 5, 7]
results = [None] * len(numbers)
threads = [threading.Thread(target=square_into, args=(n, results, i)) for i, n in enumerate(numbers)]
for t in threads: t.start()
for t in threads: t.join()
print(results)                                             # [9, 25, 49]

join returned: None
[9, 25, 49]


## 6. Subclass Thread and override run()

<img src="illustrations/06_subclass.png" alt="Downloader extends Thread. After join, size is 500 for the url ai-47." width="100%">

*Slide: Day 5 again: a Thread subclass* · script `06_thread_subclass.py`

`Downloader` is a `Thread`. `start()` is still the library method: it opens the new thread and calls your `run()`. The url and the size live on the object, so after `join()` the main thread can read `d.size`. This cell uses the url `ai-47`, and the size is `len("ai-47") * 100`, which is **500**.

`super().__init__` finishes building the thread object. Keep that line.


In [30]:
import threading, time

class Downloader(threading.Thread):
    def __init__(self, url):
        super().__init__(name=f"dl-{url}")
        self.url = url
        self.size = None

    def run(self):               # start() calls run() in the new thread
        time.sleep(0.1)
        self.size = len(self.url) * 100

d = Downloader("ai-47")
d.start()
threading.active_count()
d.join()
print(d.name, d.size)                                      # dl-iti.gov.eg 1000

dl-ai-47 500


## 7. A daemon heartbeat

<img src="illustrations/07_daemon.png" alt="Four beats of a daemon heartbeat, then main leaves and the pulse stops." width="100%">

*Slide: Daemon threads: background helpers* · script `07_daemon.py`

A daemon is a helper whose life is tied to the process. The heartbeat prints `beat` while main sleeps for a second, and the beats end when the process exits.

The notebook kernel is a long-lived process, so a daemon started in a cell would keep beating under later cells. This cell runs `07_daemon.py` as its own program. Keep the notebook in the demos folder, where that script lives. With `daemon=False` in the script, the process stays up after main is finished. Stop that version with Ctrl+C.


In [7]:
# The heartbeat would keep beating inside the notebook kernel, so this cell runs the script as its own program.
import subprocess, sys
r = subprocess.run([sys.executable, "07_daemon.py"], capture_output=True, text=True)
print(r.stdout)

beat
beat
beat
beat
main: exiting



## 8. print() can mix lines; logging does not

<img src="illustrations/08_logging.png" alt="print lines overlap. logging writes one whole line per thread and includes the thread name." width="100%">

*Slide: Clean code for threaded programs* · script `08_print_vs_logging.py`

Ten threads call `print` at the same moment, and the characters of those lines can share a row. `logging` locks each record, so every line arrives whole, and the format carries the thread name. The `worker-N` lines are that second half of the cell.


In [8]:
import threading, logging

def shout(i):
    print("hello from thread", i)

ts = [threading.Thread(target=shout, args=(i,)) for i in range(10)]
for t in ts: t.start()
for t in ts: t.join()

print("--- with logging ---")
logging.basicConfig(level=logging.INFO, format="%(threadName)-10s %(message)s", force=True)

def tell(i):
    logging.info("hello from thread %d", i)

ts = [threading.Thread(target=tell, args=(i,), name=f"worker-{i}") for i in range(4)]
for t in ts: t.start()
for t in ts: t.join()

worker-0   hello from thread 0
worker-1   hello from thread 1
worker-2   hello from thread 2
worker-3   hello from thread 3


hello from thread 0
hello from thread 1
hello from thread 2
hello from thread 3
hello from thread 4
hello from thread 5
hello from thread 6
hello from thread 7
hello from thread 8
hello from thread 9
--- with logging ---


## 9. Threads do not speed up pure-Python math (with the GIL)

<img src="illustrations/09_gil.png" alt="With the GIL, two counting threads took 0.9 seconds after a sequential run of 0.5 seconds." width="100%">

*Slide: CPU work: two threads, no faster* · script `09_gil_cpu.py`

The GIL is one lock around the bytecode interpreter. The two counting threads take turns holding it, so `n -= 1` runs one thread at a time. On this kernel the sequential pair took **0.5s** and the two threads took **0.9s**.

Run `09_gil_cpu.py` with `python3.14t`. That build can run the two counts together. This notebook prints `GIL enabled: True`.


In [9]:
import threading, time, sys

def count(n):
    while n > 0:
        n -= 1                   # pure Python, pure CPU

N = 20_000_000
start = time.perf_counter()
count(N); count(N)
print(f"sequential: {time.perf_counter() - start:.1f} s")

start = time.perf_counter()
ts = [threading.Thread(target=count, args=(N,)) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
print(f"2 threads:  {time.perf_counter() - start:.1f} s")
print("GIL enabled:", sys._is_gil_enabled())

sequential: 0.5 s
2 threads:  0.9 s
GIL enabled: True


## 10. balance += 1 in four threads

<img src="illustrations/10_balance.png" alt="Read, add, write. Four threads share one balance, and this GIL build prints 4000000." width="100%">

*Slide: The GIL is not a lock for your data* · script `10_gil_hides_race.py`

`balance += 1` is three steps: read the number, add one, write it back. Four threads do that a million times each. The total to expect is 4000000. On this GIL build those three steps often stay together, and the cell prints **4000000**.

Run the script on `python3.14t` three times. The total moves. A lock around the update is what holds it still on that build.


In [10]:
import threading

balance = 0

def deposit(times):
    global balance
    for _ in range(times):
        balance += 1             # read, add, write

ts = [threading.Thread(target=deposit, args=(1_000_000,)) for _ in range(4)]
for t in ts: t.start()
for t in ts: t.join()
print(balance)                   # expected 4000000

4000000


## 11. Two cash machines, one account

<img src="illustrations/11_race.png" alt="Both cash machines pay 80 from an account of 100, and the balance ends at -60." width="100%">

*Slide: Check, then act: the classic race* · script `11_check_then_act.py`

Both machines read a balance of 100, and both decide that 80 is allowed. The `sleep` is the gap where the other thread steps in. It stands in for any real pause between the check and the withdrawal. Both pay, `paid` is `[80, 80]`, and the account ends at **-60**.


In [11]:
import threading, time

balance = 100
paid = []

def withdraw(amount):
    global balance
    if balance >= amount:        # 1. check
        time.sleep(0.01)         #    talk to the bank
        balance -= amount        # 2. act
        paid.append(amount)

ts = [threading.Thread(target=withdraw, args=(80,)) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
print("paid out:", paid)         # [80, 80]
print("balance:", balance)       # -60

paid out: [80, 80]
balance: -60


## 12. The same account with a Lock

<img src="illustrations/12_lock.png" alt="One machine pays 80 while the other waits. The balance stays 20." width="100%">

*Slide: A Lock makes check-and-act one step* · script `12_lock_fix.py`

`with lock` holds the check and the withdrawal together. The second machine waits outside until the first has finished both steps. One payout happens. The balance stays **20**, and `paid` is `[80]`.


In [12]:
import threading, time

balance = 100
paid = []
lock = threading.Lock()

def withdraw(amount):
    global balance
    with lock:                   # one thread at a time
        if balance >= amount:
            time.sleep(0.01)
            balance -= amount
            paid.append(amount)

ts = [threading.Thread(target=withdraw, args=(80,)) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
print("paid out:", paid)         # [80]
print("balance:", balance)       # 20

paid out: [80]
balance: 20


## 13. A method that waits for itself

<img src="illustrations/13_rlock.png" alt="A Lock sticks with balance 0. An RLock lets the owner enter again and the balance becomes 20." width="100%">

*Slide: RLock: a lock you may take twice* · script `13_rlock.py`

`deposit_twice` already holds the lock, then calls `deposit`, which asks for it again. A `Lock` waits for a release that comes only after that inner call returns, so the thread is still alive at the timeout and the balance is 0. An `RLock` counts entries for its owner, the inner call goes through, and the balance becomes **20**.

The join uses a one-second timeout, which is how the cell can report the stuck thread and keep going.


In [13]:
import threading

class Account:
    def __init__(self, lock):
        self.balance = 0
        self.lock = lock
    def deposit(self, x):
        with self.lock:
            self.balance += x
    def deposit_twice(self, x):
        with self.lock:          # holds the lock...
            self.deposit(x)      # ...and asks for it again
            self.deposit(x)

for name, lock in (("Lock", threading.Lock()), ("RLock", threading.RLock())):
    a = Account(lock)
    t = threading.Thread(target=a.deposit_twice, args=(10,), daemon=True)
    t.start(); t.join(timeout=1)
    print(name, "stuck:", t.is_alive(), "balance:", a.balance)

Lock stuck: True balance: 0
RLock stuck: False balance: 20


## 14. Opposite lock order freezes; same order works

<img src="illustrations/14_deadlock.png" alt="A to B holds lock A and wants B. B to A holds lock B and wants A. Deadlock is True." width="100%">

*Slide: Deadlock: each holds what the other needs* · script `14_deadlock.py`

`A->B` takes lock A, then wants lock B. `B->A` takes lock B, then wants lock A. Each side holds the lock the other is waiting on, both threads stay alive, and the cell prints `deadlock: True`. The threads are daemons and `join` has a timeout, so the kernel receives that report.

Set `FIXED = True` and run the cell again. Both transfers take lock A first, then lock B, and both finish.


In [14]:
import sys, threading, time

FIXED = False          # change to True and run again
lock_a, lock_b = threading.Lock(), threading.Lock()

def transfer(first, second, name):
    with first:
        time.sleep(0.1)
        with second:             # waits for the other thread
            print(name, "done")

t1 = threading.Thread(target=transfer, daemon=True, args=(lock_a, lock_b, "A->B"))
t2 = threading.Thread(target=transfer, daemon=True,
                      args=(lock_a, lock_b, "B->A") if FIXED else (lock_b, lock_a, "B->A"))
t1.start(); t2.start()
t1.join(timeout=1); t2.join(timeout=1)
print("deadlock:", t1.is_alive() and t2.is_alive())

deadlock: True


## 15. Ready, set, go

<img src="illustrations/15_event.png" alt="Runners A, B, and C wait on the line until set opens the gate." width="100%">

*Slide: Event: one thread signals, many wake up* · script `15_event.py`

The event starts clear. Each runner reaches `go.wait()` and stands on the line. One `go.set()` from main releases all three. A runner that arrives after `set()` finds the gate already open.

This cell does not join the runners, so their `running` lines can appear under the next cell.


In [15]:
import threading, time

go = threading.Event()             # starts "not set"

def runner(name, delay):
    print(name, "ready")
    go.wait()                      # block until set()
    time.sleep(delay)
    print(name, "running")

for i, name in enumerate(["A", "B", "C"]):
    threading.Thread(target=runner, args=(name, 0.1 * (i + 1))).start()
    time.sleep(0.05)
print("main: ready, set...")
time.sleep(0.5)
print("main: go!")
go.set()                           # wakes every waiting thread

A ready
B ready
C ready
main: ready, set...
main: go!


## 16. Six downloads, two at a time

<img src="illustrations/16_semaphore.png" alt="Two downloads run at once. Finish times fall in waves at 0.2, 0.4, and 0.6 seconds." width="100%">

*Slide: Semaphore: at most N threads at a time* · script `16_semaphore.py`

`Semaphore(2)` is a pile of two permits. Six downloads each take one, so they finish in three waves about 0.2 seconds apart: **0.2, 0.2, 0.4, 0.4, 0.6, 0.6**.

Change `2` to `3` and run the cell again. The waves get wider. The `running` lines from the event demo can print at the top of this cell, because those runners were still going.


In [16]:
import threading, time

slots = threading.Semaphore(2)     # at most 2 at once
finished = []

def download():
    with slots:                    # take a slot, or wait for one
        time.sleep(0.2)
        finished.append(round(time.perf_counter() - start, 1))

start = time.perf_counter()
ts = [threading.Thread(target=download) for _ in range(6)]
for t in ts: t.start()
for t in ts: t.join()
print("finish times:", sorted(finished))    # [0.2, 0.2, 0.4, 0.4, 0.6, 0.6]

A running
B running
C running
finish times: [0.2, 0.2, 0.4, 0.4, 0.6, 0.6]


## 17. Producer, queue, three workers

<img src="illustrations/17_queue.png" alt="A producer hands numbered jobs to three workers. None is the signal to stop." width="100%">

*Slide: Queue: hand work from one thread to another* · script `17_queue_workers.py`

The producer puts the numbers 1 through 6 on a queue. Three workers take them, sleep a moment, and put `(n, n * n, name)` on a second queue. The handoff lives inside `Queue`. `None`, one per worker, is the signal to stop. The printout is sorted into job order, and the names show that the workers shared the pile.


In [17]:
import threading, queue, time

jobs = queue.Queue()
results = queue.Queue()

def worker():
    while True:
        n = jobs.get()             # waits for an item
        if n is None:              # sentinel: stop
            break
        time.sleep(0.1)            # do the work
        results.put((n, n * n, threading.current_thread().name))

workers = [threading.Thread(target=worker, name=f"worker-{i}") for i in range(3)]
for w in workers: w.start()
for n in range(1, 7):              # the producer
    jobs.put(n)
for _ in workers:
    jobs.put(None)                 # one stop signal each
for w in workers: w.join()
for n, sq, who in sorted(results.queue):
    print(f"{n} -> {sq:2}  by {who}")

1 ->  1  by worker-0
2 ->  4  by worker-1
3 ->  9  by worker-2
4 -> 16  by worker-0
5 -> 25  by worker-2
6 -> 36  by worker-1


## 18. ThreadPoolExecutor.map

<img src="illustrations/18_pool_map.png" alt="a.csv, b.csv, and c.csv go into pool.map and come back in that same order." width="100%">

*Slide: pool.map: the threaded version of map()* · script `18_pool_map.py`

The pool is the hand-built thread crew from demo 2, with the bookkeeping done for you. `map` returns results in input order, so `a.csv` prints before `b.csv` even though `b.csv` is the first to finish. The wall clock is still about **1.0 second**, the slowest file.


In [18]:
from concurrent.futures import ThreadPoolExecutor
import time

def download(name, seconds):
    time.sleep(seconds)
    return f"{name}: {int(seconds * 1000)} KB"

files = {"a.csv": 1.0, "b.csv": 0.6, "c.csv": 0.8}
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    for line in pool.map(download, files, files.values()):
        print(line)
print(f"total: {time.perf_counter() - start:.1f} s")

a.csv: 1000 KB
b.csv: 600 KB
c.csv: 800 KB
total: 1.0 s


## 19. Results as they finish, failures included

<img src="illustrations/19_as_completed.png" alt="virus.exe fails at 0.2 seconds, then b.csv, c.csv, and a.csv finish." width="100%">

*Slide: submit() and as_completed()* · script `19_pool_submit.py`

`submit` hands back a future for each file. `as_completed` yields those futures as they finish, so the fast failure, `virus.exe` at 0.2 seconds, shows up first. `fut.result()` raises that `ValueError` again in the main thread, and the other three files still print.


In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def download(name, seconds):
    time.sleep(seconds)
    if name.endswith(".exe"):
        raise ValueError(f"refusing {name}")
    return f"{name}: {int(seconds * 1000)} KB"

files = {"a.csv": 1.0, "b.csv": 0.6, "virus.exe": 0.2, "c.csv": 0.8}
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(download, f, s): f for f, s in files.items()}
    for fut in as_completed(futures):     # in finishing order
        try:
            print(fut.result())
        except ValueError as e:
            print("failed:", e)

failed: refusing virus.exe
b.csv: 600 KB
c.csv: 800 KB
a.csv: 1000 KB


## 20. Plain Thread vs pool

<img src="illustrations/20_exceptions.png" alt="A plain thread loses ValueError after printing it. A future keeps it so main can catch bad data." width="100%">

*Slide: Where do a thread's exceptions go?* · script `20_thread_exceptions.py`

A plain `Thread` writes the traceback, and then there is nothing left to catch. Main goes on. A pool stores the exception inside the future, and `fut.result()` raises it in the caller. That is the line `main caught: bad data`.


In [20]:
import threading
from concurrent.futures import ThreadPoolExecutor

def task():
    raise ValueError("bad data")

t = threading.Thread(target=task)
t.start()
t.join()
print("main: carried on, and cannot catch it")

with ThreadPoolExecutor() as pool:
    fut = pool.submit(task)
try:
    fut.result()
except ValueError as e:
    print("main caught:", e)

Exception in thread Thread-45 (task):
Traceback (most recent call last):
  File "/usr/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.14/threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_43784/359172312.py", line 5, in task
    raise ValueError("bad data")
ValueError: bad data


main: carried on, and cannot catch it
main caught: bad data


## 21. ThreadPoolExecutor vs ProcessPoolExecutor

<img src="illustrations/21_processes.png" alt="Threads sharing one GIL took 1.8 seconds. Two processes took 0.6 seconds." width="100%">

*Slide: Same API, real parallelism: processes* · script `21_processes.py`

The call is the same family. The threads share this interpreter's GIL, so the two counts queue up: **1.8s** on this machine. Each process has an interpreter of its own, so the two counts run together: **0.6s**.

A process pool imports the function. The cell just above writes `work.py`, and this cell imports `count` from there. Run the notebook from the demos folder.


In [21]:
%%writefile work.py
def count(n):
    while n > 0:
        n -= 1


Writing work.py


In [22]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from work import count     # a process pool needs an importable function

for Pool in (ThreadPoolExecutor, ProcessPoolExecutor):
    start = time.perf_counter()
    with Pool(max_workers=2) as pool:
        list(pool.map(count, [40_000_000] * 2))
    print(f"{Pool.__name__:20} {time.perf_counter() - start:.1f} s")

ThreadPoolExecutor   1.8 s
ProcessPoolExecutor  0.6 s


## 22. Real web pages: sequential vs a thread pool

<img src="illustrations/22_downloads.png" alt="Six web pages took 3.2 seconds one by one and 0.6 seconds together." width="100%">

*Slide: Looking ahead* · script `22_real_downloads.py`

Same shape as the made-up downloads, with real round trips. One at a time, this run took **3.2s**. The pool overlaps the six pages and the total falls to **0.6s**, close to the slowest page.

Wikipedia answered with an HTTP error on this run. The cell reports that failure and still finishes the other pages. It needs a network. The times will move from one room to the next. The shape of the result stays.


In [23]:
import time, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

URLS = ["https://www.python.org", "https://docs.python.org/3/", "https://pypi.org",
        "https://peps.python.org", "https://www.wikipedia.org", "https://github.com"]

def fetch(url):
    with urllib.request.urlopen(url, timeout=10) as r:
        return len(r.read())

def sequential():
    for u in URLS:
        try:
            fetch(u)
        except OSError:
            pass

start = time.perf_counter()
sequential()
print(f"one at a time: {time.perf_counter() - start:.1f} s")

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(URLS)) as pool:
    futures = {pool.submit(fetch, u): u for u in URLS}
    for fut in as_completed(futures):
        try:
            print(f"{fut.result():>9,} bytes  {futures[fut]}")
        except OSError as e:
            print(f"   failed        {futures[fut]}  ({type(e).__name__})")
print(f"thread pool:   {time.perf_counter() - start:.1f} s")

one at a time: 3.2 s
   27,885 bytes  https://pypi.org
   failed        https://www.wikipedia.org  (HTTPError)
   19,632 bytes  https://docs.python.org/3/
   53,183 bytes  https://www.python.org
  413,359 bytes  https://peps.python.org
  530,043 bytes  https://github.com
thread pool:   0.6 s


## 23. Which interpreter am I running?

<img src="illustrations/23_which_python.png" alt="This kernel is Python 3.14 with the GIL on. python3.14t is the free-threaded build." width="100%">

*Slide: Python without the GIL* · script `23_which_python.py`

The cell prints the version, whether this build can run without the GIL, and whether the GIL is on right now. This kernel is **3.14** with the GIL enabled. The free-threaded build is `python3.14t`, and demos 9 and 10 are the cells whose numbers change when you run them there.

```
python3.14  23_which_python.py
python3.14t 23_which_python.py
```


In [24]:
import sys, sysconfig

ft = sysconfig.get_config_var("Py_GIL_DISABLED")
print(sys.version.split()[0])
print("free-threaded build:", bool(ft))
print("GIL enabled now:", sys._is_gil_enabled())

3.14.4
free-threaded build: False
GIL enabled now: True
